# Reciter TTS — MMS-TTS Russian (VITS) → ONNX → Android

Экспорт VITS-модели TTS (facebook/mms-tts-rus или совместимой) → ONNX → Android.

Создаёт ZIP со следующим содержимым:
  models/vits_rus_android.onnx   (input_ids, attention_mask → waveform)
  models/vocab.json              (char → id, для токенизатора на устройстве)
  models/model.json              (architecture = "vits")

Импортируйте его в приложении (Модели → Выбрать локальный ZIP). Движок VITS приложения
(architecture "vits") его запускает. ПРИМЕЧАНИЕ: MMS использует посимвольный словарь; если
токенизатор сообщает is_uroman=True, языку нужна латинизация (romanization), которую
токенизатор на устройстве пока не делает — выбирайте модель, чей словарь уже покрывает нужную письменность.

Запускать в Google Colab (GPU необязателен — VITS небольшая).

**Выполняйте сверху вниз; последняя ячейка скачивает ZIP. Импорт через Модели → Выбрать локальный ZIP.**

In [ ]:
import os
import json
import zipfile


## ЯЧЕЙКА 1: Установка зависимостей

In [ ]:
!pip install -q torch transformers onnx onnxruntime numpy

import torch
from transformers import VitsModel, VitsTokenizer

MODEL_NAME = "facebook/mms-tts-rus"
WORK_DIR = "/content/vits-onnx"
os.makedirs(WORK_DIR, exist_ok=True)


## ЯЧЕЙКА 2: Загрузка модели + токенизатора

In [ ]:
print(f"Loading {MODEL_NAME} …")
model = VitsModel.from_pretrained(MODEL_NAME).eval()
tokenizer = VitsTokenizer.from_pretrained(MODEL_NAME)

sample_rate = int(getattr(model.config, "sampling_rate", 16000))
print(f"  sample_rate = {sample_rate}")
print(f"  is_uroman   = {getattr(tokenizer, 'is_uroman', None)} "
      f"(True ⇒ needs romanization, not supported on-device)")
print(f"  vocab size  = {tokenizer.vocab_size}")


## ЯЧЕЙКА 3: Экспорт в ONNX (input_ids, attention_mask → waveform)

In [ ]:
class VitsWrapper(torch.nn.Module):
    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, input_ids, attention_mask):
        return self.m(input_ids=input_ids, attention_mask=attention_mask).waveform


wrapper = VitsWrapper(model).eval()

sample = tokenizer("привет, это проверка синтеза речи", return_tensors="pt")
dummy_ids = sample["input_ids"]
dummy_mask = sample.get("attention_mask", torch.ones_like(dummy_ids))

onnx_path = f"{WORK_DIR}/vits_rus_android.onnx"
with torch.no_grad():
    wav = wrapper(dummy_ids, dummy_mask)
    print(f"  Forward OK: waveform {tuple(wav.shape)}")
    torch.onnx.export(
        wrapper, (dummy_ids, dummy_mask), onnx_path,
        input_names=["input_ids", "attention_mask"],
        output_names=["waveform"],
        dynamic_axes={
            "input_ids": {0: "batch", 1: "seq"},
            "attention_mask": {0: "batch", 1: "seq"},
            "waveform": {0: "batch", 1: "samples"},
        },
        opset_version=17, do_constant_folding=True,
    )
print(f"  ONNX: {os.path.getsize(onnx_path) / 1024**2:.0f} MB")


## ЯЧЕЙКА 4: INT8-квантование (необязательно, меньше размер загрузки)

In [ ]:
try:
    from onnxruntime.quantization import quantize_dynamic, QuantType
    q_path = f"{WORK_DIR}/vits_rus_android_int8.onnx"
    quantize_dynamic(onnx_path, q_path, weight_type=QuantType.QInt8)
    if os.path.getsize(q_path) < os.path.getsize(onnx_path):
        os.replace(q_path, onnx_path)
        print(f"  Quantized: {os.path.getsize(onnx_path) / 1024**2:.0f} MB")
except Exception as e:
    print(f"  Quantization skipped: {e}")


## ЯЧЕЙКА 5: Запись vocab.json (char → id) для токенизатора на устройстве

In [ ]:
vocab = tokenizer.get_vocab()  # token(str) → id(int)
with open(f"{WORK_DIR}/vocab.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False)
print(f"  vocab.json: {len(vocab)} symbols (sample: {list(vocab)[:12]})")


## ЯЧЕЙКА 6: Генерация манифеста model.json (architecture = vits)

In [ ]:
manifest = {
    "schemaVersion": 1,
    "id": "mms-tts-rus",
    "displayName": "MMS-TTS Russian (VITS)",
    "family": "vits",
    "architecture": "vits",
    "sampleRateHz": sample_rate,
    "tokenizer": {"type": "vits-char", "files": ["vocab.json"]},
    "files": [
        {"filename": "vits_rus_android.onnx", "role": "VOCODER", "sizeMb": round(os.path.getsize(onnx_path) / 1024**2), "required": True}
    ],
    "voices": [
        {"id": "mms_ru", "locale": "ru-RU", "displayName": "Русский (MMS)", "speakerId": 0}
    ],
}
with open(f"{WORK_DIR}/model.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)
print("  Wrote model.json")


## ЯЧЕЙКА 7: Создание ZIP + скачивание

In [ ]:
zip_path = f"{WORK_DIR}/mms-tts-rus-android.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(onnx_path, "models/vits_rus_android.onnx")
    zf.write(f"{WORK_DIR}/vocab.json", "models/vocab.json")
    zf.write(f"{WORK_DIR}/model.json", "models/model.json")
print(f"Archive: {os.path.getsize(zip_path) / 1024**2:.0f} MB")

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print(f"Archive saved to: {zip_path}")
